In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")

df = df[['country', 'year', 'co2', 'gdp', 'population', 'energy_per_capita']]
df = df.dropna()

print(df.describe())

# Feature Engineering
df['gdp_per_capita'] = df['gdp'] / df['population']

X = df[['gdp', 'population', 'energy_per_capita', 'gdp_per_capita']]
y = df['co2']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear Regression (baseline model)
lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)

lin_preds = lin_model.predict(X_test_scaled)

print("\n--- Linear Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, lin_preds)))
print("R^2:", r2_score(y_test, lin_preds))

# Polynomial Regression (captures nonlinear relationships)
poly = PolynomialFeatures(degree=2)

X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

poly_preds = poly_model.predict(X_test_poly)

print("\n--- Polynomial Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, poly_preds)))
print("R^2:", r2_score(y_test, poly_preds))

# Ridge Regression (L2 regularization to reduce overfitting)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

ridge_preds = ridge.predict(X_test_scaled)

print("\n--- Ridge Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, ridge_preds)))
print("R^2:", r2_score(y_test, ridge_preds))

# Lasso Regression (L1 regularization, feature selection)
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

lasso_preds = lasso.predict(X_test_scaled)

print("\n--- Lasso Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_preds)))
print("R^2:", r2_score(y_test, lasso_preds))

# Gradient Descent (manual linear regression implementation)
X_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
y_b = y_train.values.reshape(-1, 1)

theta = np.zeros((X_b.shape[1], 1))
learning_rate = 0.01
iterations = 1000

for i in range(iterations):
    gradients = (2 / len(X_b)) * X_b.T.dot(X_b.dot(theta) - y_b)
    theta -= learning_rate * gradients

# Predictions for GD model
X_test_b = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]
gd_preds = X_test_b.dot(theta).flatten()

print("\n--- Gradient Descent Linear Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, gd_preds)))
print("R^2:", r2_score(y_test, gd_preds))

print("\n================ SUMMARY ================")
print("Linear R2:", r2_score(y_test, lin_preds))
print("Polynomial R2:", r2_score(y_test, poly_preds))
print("Ridge R2:", r2_score(y_test, ridge_preds))
print("Lasso R2:", r2_score(y_test, lasso_preds))
print("GD R2:", r2_score(y_test, gd_preds))

              year           co2           gdp    population  \
count  7776.000000   7776.000000  7.776000e+03  7.776000e+03   
mean   1997.890947    220.971961  5.778329e+11  5.133015e+07   
std      15.139690   1472.913626  4.297781e+12  3.135671e+08   
min    1965.000000      0.022000  1.642060e+08  6.408200e+04   
25%    1986.000000      3.655500  1.796476e+10  3.573031e+06   
50%    1999.000000     20.853000  6.427934e+10  9.762675e+06   
75%    2011.000000     85.792000  2.554639e+11  2.829419e+07   
max    2022.000000  37527.773000  1.301126e+14  8.021407e+09   

       energy_per_capita  
count        7776.000000  
mean        25023.837213  
std         34284.496142  
min             0.000000  
25%          2498.382250  
50%         11540.712000  
75%         34802.650500  
max        318559.688000  

--- Linear Regression ---
RMSE: 303.16307300117234
R^2: 0.9353185446794916

--- Polynomial Regression ---
RMSE: 70.99768948792382
R^2: 0.9964525554754371

--- Ridge Regression ---